# P3-4 · Hệ sinh thái Hugging Face

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q datasets pyvi

## Lý thuyết

In [ ]:
# Cần mạng: tải model từ Hugging Face (chạy trên Colab); thay <tên-model-trên-Hub> bằng model bạn đã đẩy lên
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="<tên-model-trên-Hub>")
clf("Giảng viên dạy rất dễ hiểu")      # [{'label': ..., 'score': ...}]

In [ ]:
# Cần mạng: tải PhoBERT từ Hugging Face (chạy trên Colab)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=3,
    id2label={0: "tiêu cực", 1: "trung tính", 2: "tích cực"},
    label2id={"tiêu cực": 0, "trung tính": 1, "tích cực": 2})

In [ ]:
# Cần tokenizer PhoBERT ở ô trên (chạy trên Colab)
batch = tok(["Thầy dạy hay", "Phòng_học nóng quá , không"],
            padding=True, truncation=True, max_length=256, return_tensors="pt")
batch["input_ids"]        # (2, T): id token, câu ngắn được đệm bằng pad_token_id
batch["attention_mask"]   # (2, T): 1 = token thật, 0 = đệm

In [ ]:
from pyvi import ViTokenizer
ViTokenizer.tokenize("Phòng học nóng quá, không bật được điều hòa")

'Phòng học nóng quá , không bật được điều hòa'


In [ ]:
# Cần mạng: tải dataset và tokenizer từ Hugging Face (chạy trên Colab)
from datasets import load_dataset
ds = load_dataset("uitnlp/vietnamese_students_feedback")        # DatasetDict: train / validation / test
ds["train"][0]                                                  # một dòng dạng dict
ds["train"].features                                            # kiểu từng cột, tên nhãn

def preprocess(batch):
    return tok(batch["sentence"], truncation=True, max_length=256)
ds = ds.map(preprocess, batched=True)                           # token hóa, cache trên đĩa
ds = ds.rename_column("sentiment", "labels")                    # Trainer tìm cột tên "labels"
ds = ds.remove_columns(["topic"])                               # bỏ cột không dùng

In [ ]:
# Cần GPU và các ô trên: chạy trên Colab (T4, khoảng 10 phút)
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    "phobert-sentiment",
    learning_rate=2e-5, per_device_train_batch_size=32, num_train_epochs=3,
    warmup_steps=0.1, weight_decay=0.01, fp16=True,     # số < 1 nghĩa là tỉ lệ (10% số bước)
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    train_sampling_strategy="group_by_length", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()
trainer.evaluate(ds["test"])          # đo test một lần, ở cuối

In [ ]:
# Cần tài khoản Hugging Face: chạy trên Colab sau khi train
from huggingface_hub import login
login()                                   # token có quyền write, lấy ở trang Settings → Access Tokens
trainer.push_to_hub(commit_message="PhoBERT cảm xúc, macro F1 0.xx trên test")

## Bài tập 4: compute_metrics {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

## Bài tập 5: Tokenizer của riêng bạn: padding, truncation, attention mask {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from transformers import PreTrainedTokenizerFast
corpus = ["sản phẩm rất tốt giao hàng nhanh", "hàng bị lỗi không dùng được", "đóng gói cẩn thận sẽ mua lại",
          "giao hàng chậm sản phẩm bình thường", "rất hài lòng với chất lượng", "không đáng tiền thất vọng"] * 5
tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
tok.normalizer = normalizers.Sequence([normalizers.NFC(), normalizers.Lowercase()])
tok.pre_tokenizer = pre_tokenizers.Whitespace()
tok.train_from_iterator(corpus, trainers.WordPieceTrainer(vocab_size=120, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"]))
hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok, pad_token="[PAD]", unk_token="[UNK]")
cau = ["Rất tốt", "Giao hàng nhanh, đóng gói cẩn thận", "sản phẩm rất tốt giao hàng nhanh đóng gói cẩn thận sẽ mua lại lần sau"]
# TODO: batch = hf_tok(...); in batch["input_ids"].shape và batch["attention_mask"].sum(1)

In [ ]:
batch = hf_tok(cau, padding=True, truncation=True, max_length=10, return_tensors="pt")
print(batch["input_ids"].shape)
print(batch["attention_mask"])
for i in range(3):
    print(hf_tok.convert_ids_to_tokens(batch["input_ids"][i]))

torch.Size([3, 10])
tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
['rất', 'tố', '##t', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['giao', 'hàng', 'nh', '##anh', '[UNK]', 'đó', '##ng', 'gó', '##i', 'cẩ']
['sản', 'phẩm', 'rất', 'tố', '##t', 'giao', 'hàng', 'nh', '##anh', 'đó']


## Bài tập 6: Chuẩn bị bộ đánh giá với datasets {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from datasets import Dataset
raw = {"text": ["Rất tốt!", "", "Giao chậm quá", "Bình thường", "Đóng gói đẹp, sẽ mua lại", "Hàng lỗi", "   ",
                 "Tuyệt vời", "Không như mô tả", "Ổn", "Chất lượng cao", "Thất vọng"],
       "stars": [5, 4, 2, 3, 5, 1, 5, 5, 1, 3, 4, 2]}
ds = Dataset.from_dict(raw)
# TODO: lọc, gán nhãn, chia, thêm cột n_chars

In [ ]:
clean = (ds.filter(lambda ex: ex["text"].strip() != "" and ex["stars"] != 3)
           .map(lambda ex: {"label": int(ex["stars"] >= 4), "text": ex["text"].strip().lower()}))
split = clean.train_test_split(test_size=0.25, seed=0)
split = split.map(lambda b: {"n_chars": [len(t) for t in b["text"]]}, batched=True)
print(clean)
print(split["train"][:3])

Dataset({
    features: ['text', 'stars', 'label'],
    num_rows: 8
})
{'text': ['hàng lỗi', 'chất lượng cao', 'không như mô tả'], 'stars': [1, 4, 1], 'label': [0, 1, 0], 'n_chars': [8, 14, 15]}


## Bài tập 7: Trainer chạy trọn vòng với model tí hon {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from transformers import (PreTrainedTokenizerFast, BertConfig, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
tot = ["sản phẩm rất tốt", "giao hàng nhanh", "rất hài lòng", "đóng gói cẩn thận", "chất lượng tuyệt vời", "sẽ mua lại"]
xau = ["hàng bị lỗi", "giao hàng chậm", "rất thất vọng", "không đáng tiền", "chất lượng kém", "không mua lại"]
texts = [a + " " + b for a in tot for b in tot[:5]] + [a + " " + b for a in xau for b in xau[:5]]
labels = [1] * 30 + [0] * 30
tok = Tokenizer(models.WordPiece(unk_token="[UNK]")); tok.pre_tokenizer = pre_tokenizers.Whitespace()
tok.normalizer = normalizers.Lowercase()
tok.train_from_iterator(texts, trainers.WordPieceTrainer(vocab_size=100, special_tokens=["[PAD]", "[UNK]"]))
hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok, pad_token="[PAD]", unk_token="[UNK]")
ds = Dataset.from_dict({"text": texts, "label": labels}).shuffle(seed=0).train_test_split(test_size=0.2, seed=0)
ds = ds.map(lambda b: hf_tok(b["text"], truncation=True, max_length=16), batched=True)
# TODO: config, model (from_config), args, compute_metrics, trainer.train(), ket_qua = trainer.evaluate()

In [ ]:
config = BertConfig(vocab_size=hf_tok.vocab_size, hidden_size=32, num_hidden_layers=2, num_attention_heads=2,
                    intermediate_size=64, num_labels=2)
model = AutoModelForSequenceClassification.from_config(config)

def compute_metrics(p):
    logits, labels = p
    return {"accuracy": float((np.argmax(logits, -1) == labels).mean())}

args = TrainingArguments(output_dir="tiny-out", num_train_epochs=3, per_device_train_batch_size=8,
                         learning_rate=5e-3, eval_strategy="epoch", save_strategy="no", report_to=[],
                         logging_strategy="epoch", seed=0, use_cpu=True)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                  data_collator=DataCollatorWithPadding(hf_tok), compute_metrics=compute_metrics)
trainer.train()
ket_qua = trainer.evaluate()
print("số bước:", trainer.state.global_step, "| accuracy:", ket_qua["eval_accuracy"])

{'loss': '0.7011', 'grad_norm': '0.8469', 'learning_rate': '0.003611', 'epoch': '1'}
{'eval_loss': '0.662', 'eval_accuracy': '0.6667', 'eval_runtime': '0.0062', 'eval_samples_per_second': '1925', 'eval_steps_per_second': '320.8', 'epoch': '1'}
{'loss': '0.6193', 'grad_norm': '1.148', 'learning_rate': '0.001944', 'epoch': '2'}
{'eval_loss': '0.61', 'eval_accuracy': '0.5833', 'eval_runtime': '0.0047', 'eval_samples_per_second': '2537', 'eval_steps_per_second': '422.8', 'epoch': '2'}
{'loss': '0.5266', 'grad_norm': '1.42', 'learning_rate': '0.0002778', 'epoch': '3'}
{'eval_loss': '0.5756', 'eval_accuracy': '0.8333', 'eval_runtime': '0.005', 'eval_samples_per_second': '2388', 'eval_steps_per_second': '397.9', 'epoch': '3'}
{'train_runtime': '0.1415', 'train_samples_per_second': '1018', 'train_steps_per_second': '127.2', 'train_loss': '0.6157', 'epoch': '3'}
số bước: 18 | accuracy: 0.8333333333333334


## Bài tập 8: Lưu, nạp lại và kiểm tra model {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import os, torch
from transformers import BertConfig, AutoModelForSequenceClassification
torch.manual_seed(0)
config = BertConfig(vocab_size=100, hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                    num_labels=3, id2label={0: "tiêu cực", 1: "trung tính", 2: "tích cực"})
model = AutoModelForSequenceClassification.from_config(config).eval()
x = torch.randint(0, 100, (2, 7))
# TODO: lưu, liệt kê file, nạp lại (model2), so logits, đọc nhãn dự đoán bằng config.id2label

In [ ]:
model.save_pretrained("my-model")
print(sorted(os.listdir("my-model")))
model2 = AutoModelForSequenceClassification.from_pretrained("my-model").eval()
with torch.no_grad():
    l1, l2 = model(x).logits, model2(x).logits
giong = torch.equal(l1, l2)
print("giống hệt:", giong, "| dự đoán:", [model2.config.id2label[int(i)] for i in l2.argmax(-1)])

['config.json', 'model.safetensors']
giống hệt: True | dự đoán: ['tiêu cực', 'tích cực']
